# IELTS Writing Model Evaluation

Compare AES and six Qwen prompt setups on the same test essays. Attach the matching private Kaggle Dataset and enable a GPU and internet access for inference.


In [ ]:
"""Standard-library-only Kaggle bootstrap and verified result restoration."""
from typing import Any

import hashlib
import json
from pathlib import Path, PurePosixPath
import re
import shutil
from uuid import uuid4
import zipfile


def sha256_file(path: Any) -> Any:
    """Sha256 file.

    Args:
        path (Any): Filesystem location to read, validate or write.

    Returns:
        Any: The computed result for the supplied inputs.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def inventory_id(files: Any) -> Any:
    """Inventory id.

    Args:
        files (Any): Files used by this operation.

    Returns:
        Any: The computed result for the supplied inputs.
    """
    value = {"schema_version": "model-evaluation-bundle-v1", "files": files}
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":")).encode()).hexdigest()


def safe_relative(name: Any) -> Any:
    """Safe relative.

    Args:
        name (Any): Name used by this operation.

    Returns:
        Any: The computed result for the supplied inputs.
    """
    path = PurePosixPath(name)
    if not name or path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name or str(path) != name:
        raise ValueError(f"Unsafe archive/inventory path: {name}")
    return path


def validate_inventory(files: Any) -> None:
    """Validate inventory.

    Args:
        files (Any): Files used by this operation.

    Returns:
        None: This operation returns no value.
    """
    if not isinstance(files, dict) or not files:
        raise ValueError("Missing file inventory")
    for name, digest in files.items():
        safe_relative(name)
        if not isinstance(digest, str) or not re.fullmatch("[0-9a-f]{64}", digest):
            raise ValueError("Invalid file digest")


def copy_verified(source: Any, destination: Any, digest: Any) -> None:
    """Copy verified.

    Args:
        source (Any): Source used by this operation.
        destination (Any): Destination used by this operation.
        digest (Any): Digest used by this operation.

    Returns:
        None: This operation returns no value.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("xb") as output:
        shutil.copyfileobj(source, output, length=1024 * 1024)
    if sha256_file(destination) != digest:
        raise ValueError(f"File hash mismatch: {destination.name}")


def extract_verified(archive: Any, destination: Any, files: Any, prefix: str="") -> None:
    """Extract verified.

    Args:
        archive (Any): Archive used by this operation.
        destination (Any): Destination used by this operation.
        files (Any): Files used by this operation.
        prefix (str): Prefix used by this operation.

    Returns:
        None: This operation returns no value.
    """
    expected = {prefix + name for name in files}
    actual = [name for name in archive.namelist() if not name.endswith("/")]
    if len(actual) != len(set(actual)) or set(actual) != expected:
        raise ValueError("Archive members differ from inventory")
    for name, digest in files.items():
        safe_relative(name)
        info = archive.getinfo(prefix + name)
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlinks are not allowed")
        with archive.open(info) as source:
            copy_verified(source, destination / name, digest)


def prepare_project(input_root: Any, work_root: Any, manifest_path: Any=None, expected_bundle_id: Any=None) -> Any:
    """Prepare project.

    Args:
        input_root (Any): Input root used by this operation.
        work_root (Any): Work root used by this operation.
        manifest_path (Any): Manifest path used by this operation.
        expected_bundle_id (Any): Expected bundle id used by this operation.

    Returns:
        Any: The computed result for the supplied inputs.
    """
    candidates = [Path(manifest_path)] if manifest_path is not None else list(Path(input_root).rglob("ModelEvaluationBundleManifest.json"))
    if len(candidates) != 1:
        raise ValueError(f"Attach one new evaluation Dataset or select manifest_path explicitly; found {len(candidates)}")
    path = candidates[0]
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "model-evaluation-bundle-v1":
        raise ValueError("This is not a model evaluation Dataset")
    validate_inventory(manifest["files"])
    bundle_id = inventory_id(manifest["files"])
    if manifest.get("bundle_id") != bundle_id or (expected_bundle_id is not None and expected_bundle_id != bundle_id):
        raise ValueError("Notebook and evaluation Dataset belong to different bundles")
    project = Path(work_root).resolve() / ("evaluation-project-" + bundle_id[:12] + "-" + uuid4().hex[:10])
    project.mkdir(parents=True, exist_ok=False)
    archive_name = manifest.get("archive_name", "model_evaluation_bundle.zip")
    safe_relative(archive_name)
    archive_path = path.parent / archive_name
    if archive_path.is_file():
        if sha256_file(archive_path) != manifest["archive_sha256"]:
            raise ValueError("Evaluation archive hash mismatch")
        with zipfile.ZipFile(archive_path) as archive:
            extract_verified(archive, project, manifest["files"])
    else:
        anchor = "code/shared/paths.yaml" if "code/shared/paths.yaml" in manifest["files"] else next(iter(manifest["files"]))
        depth = len(PurePosixPath(anchor).parts)
        roots = {match.parents[depth - 1].resolve() for match in path.parent.rglob(anchor)}
        roots = [candidate for candidate in roots if all((candidate / name).is_file() for name in manifest["files"])]
        if len(roots) != 1:
            raise ValueError(f"Expected one complete unpacked evaluation input tree; found {len(roots)}")
        input_files_root = roots[0]
        for name, digest in manifest["files"].items():
            source_path = (input_files_root / name).resolve()
            if not source_path.is_relative_to(input_files_root):
                raise ValueError("Input file escapes Dataset directory")
            with source_path.open("rb") as source:
                copy_verified(source, project / name, digest)
    (project / "ModelEvaluationBundleManifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return project


def restore_export(archive_path: Any, work_root: Any) -> Any:
    """Restore export.

    Args:
        archive_path (Any): Archive path used by this operation.
        work_root (Any): Work root used by this operation.

    Returns:
        Any: The computed result for the supplied inputs.
    """
    with zipfile.ZipFile(archive_path) as archive:
        manifests = [name for name in archive.namelist() if name.endswith("/export_manifest.json")]
        if len(manifests) != 1:
            raise ValueError("Expected exactly one evaluation export manifest")
        manifest_name = manifests[0]
        safe_relative(manifest_name)
        manifest = json.loads(archive.read(manifest_name))
        if manifest.get("schema_version") != "model-evaluation-export-v1":
            raise ValueError("Unsupported evaluation export")
        files = manifest["files"]
        validate_inventory(files)
        prefix = manifest_name.removesuffix("export_manifest.json")
        expected = {prefix + name for name in files} | {manifest_name}
        if len(archive.namelist()) != len(set(archive.namelist())) or set(archive.namelist()) != expected:
            raise ValueError("Unexpected files in evaluation export")
        restored = Path(work_root).resolve() / ("restored-run-" + uuid4().hex)
        restored.mkdir(parents=True, exist_ok=False)
        for name, digest in files.items():
            info = archive.getinfo(prefix + name)
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not allowed")
            with archive.open(info) as source:
                copy_verified(source, restored / name, digest)
        (restored / "restored_export_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return restored


## 1. Select input and resume
Attach the matching new Dataset. Leave the manifest unset if exactly one evaluation Dataset is attached.
A new project directory is created each time this cell runs.
For a later session, attach an exported result ZIP and set RESUME_ARCHIVE.
Use one notebook/session as writer for each run.


In [ ]:
EXPECTED_BUNDLE_ID = None
MANIFEST_PATH = None
RESUME_ARCHIVE = None
RESUME_RUN = None  # Existing writable run directory in this session, or restored below.
project_root = prepare_project('/kaggle/input', '/kaggle/working/model-evaluation-projects',
                               MANIFEST_PATH, EXPECTED_BUNDLE_ID)
if RESUME_ARCHIVE is not None:
    RESUME_RUN = restore_export(RESUME_ARCHIVE, '/kaggle/working/model-evaluation-restored')
print('Project:', project_root)
print('Resume:', RESUME_RUN)


## 2. Install the pinned evaluation dependencies
A new virtual environment inherits the host CUDA PyTorch; pinned evaluation packages are installed into the new environment.
The exact torch constraint preserves the installed CUDA build. All model work runs in a fresh Python subprocess.
Package conflicts stop installation rather than silently changing the requested versions.


In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import venv
env_path = project_root / '.evaluation-venv'
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(env_path)
EVALUATION_PYTHON = str(env_path / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
constraint = project_root / 'torch-constraint.txt'
constraint.write_text('torch==' + importlib.metadata.version('torch') + '\n', encoding='utf-8')
subprocess.check_call([sys.executable, '-m', 'pip', '--python', EVALUATION_PYTHON, 'install', '-r',
                       str(project_root / 'code/offline_research/requirements-evaluation.txt'), '-c', str(constraint)])
os.environ['HF_HOME'] = '/kaggle/working/model-evaluation-cache/huggingface'
os.environ['HF_HUB_CACHE'] = os.environ['HF_HOME'] + '/hub'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'


## 3. Experiment and plot configuration
Edit the YAML below. Each changed experiment receives a new run.
For a technical pilot set sample_size to 10 or 20 before starting; the default is the full test pool.
`report.plots` chooses graph types; `[]` still exports all numeric tables. There is no smoothing or half-band accuracy.
Changing only report settings permits re-reporting an existing run into a new report directory.


In [ ]:
CONFIG_YAML = (project_root / 'code/offline_research/notebooks/ModelEvaluationConfig.yaml').read_text(encoding='utf-8')
print(CONFIG_YAML)
# Replace CONFIG_YAML with an edited triple-quoted YAML string if needed.
# Plot types: metric_comparison, metric_bars, metric_heatmap, calibration, calibration_by_prediction, bias,
# error_histogram, error_histogram_std, error_ridge, confusion_matrix, score_distribution,
# coverage, prompt_comparison, repeatability, latency.


## 4. Run models or regenerate reports
Modes: all, prepare (real AES/BGE only), infer (needs RESUME_RUN), report (no model loading; needs RESUME_RUN).
MAX_JOBS optionally limits NEW judge jobs per invocation. Failed jobs remain recorded; resume runs only missing jobs.
The command prints the run path: set RESUME_RUN to that path before continuing it.
The full default matrix is 1,635 essays x 6 setups = 9,810 judge jobs.


In [ ]:
MODE = 'all'
MAX_JOBS = None
config_path = project_root / ('evaluation-config-' + uuid4().hex + '.yaml')
config_path.write_text(CONFIG_YAML, encoding='utf-8')
command = [EVALUATION_PYTHON, '-B', str(project_root / 'code/offline_research/EvaluateModels.py'),
           '--config', str(config_path), '--mode', MODE, '--export',
           '--output-root', '/kaggle/working/model-evaluation-runs']
if RESUME_RUN is not None:
    command += ['--resume-run', str(RESUME_RUN)]
if MAX_JOBS is not None:
    command += ['--max-jobs', str(MAX_JOBS)]
subprocess.check_call(command)


## 5. Download results
Download the printed ZIP with predictions, metrics and charts. Timings describe model inference on Kaggle.


## 6. Calibration curves and a separate results ZIP
This cell exports existing results without running models. The archive is placed in /kaggle/working/IELTS_Evaluation_Results with a readable name.


In [ ]:
"""Readable exports and notebook previews, including existing evaluation runs."""
from typing import Any

from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import zipfile


def export_results(run: Any, destination: Any=None) -> Any:
    """Archive an existing evaluation run with a file inventory.

    Args:
        run (Any): Run directory containing run_manifest.json.
        destination (Any): ZIP output directory. Defaults to the run's exports subdirectory.

    Returns:
        Any: Path to the new ZIP. A numeric suffix avoids overwriting an earlier export.
    """
    run = Path(run).resolve()
    destination = Path(destination).resolve() if destination is not None else run / "exports"
    destination.mkdir(parents=True, exist_ok=True)
    manifest = json.loads((run / "run_manifest.json").read_text(encoding="utf-8"))
    plan = manifest.get("plan", {})
    models = "_".join(plan.get("model_labels", {}).values()) or "models"
    models = re.sub(r"[^a-zA-Z0-9.-]+", "_", models).strip("_")[:100]
    stamp = re.match(r"(\d{4})(\d{2})(\d{2})T(\d{2})(\d{2})(\d{2})Z", run.name)
    date = "-".join(stamp.groups()[:3]) + "_" + "-".join(stamp.groups()[3:]) if stamp else datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
    stem = f"IELTS_Evaluation_{models}_{len(plan.get('essay_ids', []))}_essays_{len(plan.get('setups', []))}_prompts_{plan.get('repeats', 1)}_repeat_{date}"
    archive = destination / (stem + ".zip")
    number = 2
    while archive.exists():
        archive = destination / f"{stem}_{number}.zip"
        number += 1
    files = [candidate_path for candidate_path in sorted(run.rglob("*")) if candidate_path.is_file() and "exports" not in candidate_path.relative_to(run).parts
             and not candidate_path.resolve().is_relative_to(destination) and not candidate_path.name.endswith(".tmp") and candidate_path.name != "export_manifest.json"]
    def digest(path: Any) -> str:
        """Hash an exported file in chunks.

        Args:
            path (Any): Path of a file to include in the inventory.

        Returns:
            str: SHA-256 hex digest of the file contents.
        """
        value = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                value.update(chunk)
        return value.hexdigest()
    inventory = {filename.relative_to(run).as_posix(): digest(filename) for filename in files}
    with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as output:
        for path in files:
            output.write(path, run.name + "/" + path.relative_to(run).as_posix())
        output.writestr(run.name + "/export_manifest.json", json.dumps({"schema_version": "model-evaluation-export-v1", "files": inventory}, indent=2))
    return archive


def show_results(run: Any, destination: str="/kaggle/working/IELTS_Evaluation_Results") -> Any:
    """Export a saved run and display calibration plots from its latest report.

    Args:
        run (Any): Run directory containing run_manifest.json.
        destination (str): Directory where the ZIP will be written.

    Returns:
        Any: Path to the new ZIP, including when no report is available.
    """
    from IPython.display import display, Markdown, FileLink, Image
    run = Path(run).resolve()
    archive = export_results(run, destination)
    display(Markdown(f"**Готовый ZIP:** `{archive.name}`\n\nЗапуск: `{run}`"))
    display(FileLink(os.path.relpath(archive, Path.cwd())))
    reports = list((run / "reports").glob("*/report_manifest.json"))
    if not reports:
        display(Markdown("Отчёт ещё не создан; архив содержит сохранённые результаты."))
        return archive
    report = max(reports, key=lambda candidate_path: candidate_path.stat().st_mtime).parent
    display(Markdown("**Графики:** adjacent accuracy — доля оценок в пределах ±1 band; "
        "MAE/RMSE — ошибка; Pearson/Spearman — корреляция; QWK — согласие с оценками датасета; "
        "calibration — средний прогноз при заданном эталонном балле. Ниже показаны calibration curves."))
    images = sorted((report / "plots").glob("*.png"))
    curves = [candidate_path for candidate_path in images if "score_response" in candidate_path.name
              or re.search(r"_calibration_(?:grid|TR|CC|LR|GRA)\.png$", candidate_path.name)]
    if not curves:
        display(Markdown("Calibration curves не найдены. Проверьте выбранные типы графиков в конфиге отчёта."))
    for path in curves:
        display(Markdown(f"**Calibration curves:** `{path.name}`"))
        display(Image(filename=str(path)))
    display(Markdown("**Все изображения в отчёте:**"))
    for path in images:
        display(FileLink(os.path.relpath(path, Path.cwd())))
    return archive


RESULTS_RUN = RESUME_RUN
if RESULTS_RUN is None:
    candidates = list(Path('/kaggle/working/model-evaluation-runs').rglob('run_manifest.json'))
    if not candidates:
        raise ValueError('Set RESULTS_RUN to the printed Run directory')
    RESULTS_RUN = max(candidates, key=lambda p: p.stat().st_mtime).parent
show_results(RESULTS_RUN)
